In [1]:
import copy

def read_board(file):
    board = []
    with open(file, 'r') as f:
        for line in f:
            board.append([int(x) for x in line.strip()])
    return board

def print_board(board):
    for row in board:
        print(" ".join(str(x) for x in row))

def get_domains(board):
    domains = {}
    for r in range(9):
        for c in range(9):
            if board[r][c] == 0:
                domains[(r,c)] = set(range(1,10))
            else:
                domains[(r,c)] = {board[r][c]}
    return domains

def neighbors(cell):
    r, c = cell
    n = set()
    for i in range(9):
        n.add((r, i))
        n.add((i, c))
    br, bc = 3*(r//3), 3*(c//3)
    for i in range(br, br+3):
        for j in range(bc, bc+3):
            n.add((i,j))
    n.remove(cell)
    return n

def revise(domains, xi, xj):
    revised = False
    for x in set(domains[xi]):
        if all(x == y for y in domains[xj]):
            domains[xi].remove(x)
            revised = True
    return revised

def ac3(domains):
    queue = [(xi, xj) for xi in domains for xj in neighbors(xi)]
    while queue:
        xi, xj = queue.pop(0)
        if revise(domains, xi, xj):
            if len(domains[xi]) == 0:
                return False
            for xk in neighbors(xi):
                if xk != xj:
                    queue.append((xk, xi))
    return True

def is_complete(domains):
    return all(len(domains[v]) == 1 for v in domains)

def select_var(domains):
    unassigned = [v for v in domains if len(domains[v]) > 1]
    return min(unassigned, key=lambda v: len(domains[v]))

def forward_check(domains, var, value):
    new_domains = copy.deepcopy(domains)
    new_domains[var] = {value}
    for n in neighbors(var):
        if value in new_domains[n]:
            new_domains[n].remove(value)
            if len(new_domains[n]) == 0:
                return None
    return new_domains

calls = 0
fails = 0

def backtrack(domains):
    global calls, fails
    calls += 1
    if is_complete(domains):
        return domains
    var = select_var(domains)
    for val in domains[var]:
        new_domains = forward_check(domains, var, val)
        if new_domains is not None:
            if ac3(new_domains):
                result = backtrack(new_domains)
                if result:
                    return result
    fails += 1
    return None

def solve(file):
    global calls, fails
    calls, fails = 0, 0

    print("-----------------------------------\n")
    print("Board:", file)
    print("-----------------------------------\n")

    board = read_board(file)
    domains = get_domains(board)

    ac3(domains)
    result = backtrack(domains)

    if result:
        solved = [[0]*9 for _ in range(9)]
        for (r,c) in result:
            solved[r][c] = list(result[(r,c)])[0]

        print("Solved Board:\n")
        print_board(solved)
    else:
        print("No solution found :(")
        print("-----------------------------------\n")

    print("\nBacktrack Calls:", calls)
    print("Backtrack Failures:", fails)


files = [
    "E:\\5th Sem\\AI\\23F-0855_A5_6C\\easy.txt",
    "E:\\5th Sem\\AI\\23F-0855_A5_6C\\medium.txt",
    "E:\\5th Sem\\AI\\23F-0855_A5_6C\\hard.txt",
    "E:\\5th Sem\\AI\\23F-0855_A5_6C\\_veryhard.txt"
]

for f in files:
    solve(f)
    print("-----------------------------------\n")



-----------------------------------

Board: E:\5th Sem\AI\23F-0855_A5_6C\easy.txt
-----------------------------------

Solved Board:

7 8 4 9 3 2 1 5 6
6 1 9 4 8 5 3 2 7
2 3 5 1 7 6 4 8 9
5 7 8 2 6 1 9 3 4
3 4 1 8 9 7 5 6 2
9 2 6 5 4 3 8 7 1
4 5 3 7 2 9 6 1 8
8 6 2 3 1 4 7 9 5
1 9 7 6 5 8 2 4 3

Backtrack Calls: 1
Backtrack Failures: 0
-----------------------------------

-----------------------------------

Board: E:\5th Sem\AI\23F-0855_A5_6C\medium.txt
-----------------------------------

Solved Board:

8 7 5 9 3 6 1 4 2
1 6 9 7 2 4 3 8 5
2 4 3 8 5 1 6 7 9
4 5 2 6 9 7 8 3 1
9 8 6 4 1 3 2 5 7
7 3 1 5 8 2 9 6 4
5 1 7 3 6 9 4 2 8
6 2 8 1 4 5 7 9 3
3 9 4 2 7 8 5 1 6

Backtrack Calls: 2
Backtrack Failures: 0
-----------------------------------

-----------------------------------

Board: E:\5th Sem\AI\23F-0855_A5_6C\hard.txt
-----------------------------------

Solved Board:

1 5 2 3 4 6 8 9 7
4 3 7 1 8 9 6 5 2
6 8 9 5 7 2 3 1 4
8 2 1 6 3 7 9 4 5
5 4 3 8 9 1 7 2 6
9 7 6 4 2 5 1 8 3
7 9 8 